# Experiment 1: layer-wise LLaVA vision representations

This experiment compares the source $x_s$, target $x_t$, and AMP adversarial source $x_{adv}$ at every hidden state returned by the LLaVA-1.5-7B vision tower. It follows `attack_llava.py`: the same model, FP16 dtype, 336×336 bicubic resize, CLIP normalization, and complete token tensor (no pooling or token removal). Hidden-state tensors are flattened only when computing cosine similarity.

In [ ]:
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision import transforms
from transformers import LlavaForConditionalGeneration

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
DTYPE = torch.float16
DEVICE = torch.device("cuda")
IMAGE_PATHS = {
    "source": Path("../source_img.png"),
    "target": Path("../target_img.png"),
    "adversarial": Path("./amp-llava.png"),
}
OUTPUT_CSV = Path("./exp1_layerwise_cosine.csv")

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required, matching attack_llava.py.")
missing = [str(path) for path in IMAGE_PATHS.values() if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing experiment image(s): {missing}")

In [ ]:
# Match AMP's model-loading and preprocessing conventions exactly.
llava_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
vision_tower = llava_model.vision_tower.to(DEVICE).eval()
del llava_model  # Keep only the component used by AMP for this experiment.

to_tensor = transforms.ToTensor()
preprocess = transforms.Compose(
    [
        transforms.Resize(
            (336, 336),
            interpolation=transforms.InterpolationMode.BICUBIC,
        ),
        transforms.Normalize(
            (0.48145466, 0.4578275, 0.40821073),
            (0.26862954, 0.26130258, 0.27577711),
        ),
    ]
)

In [ ]:
def extract_hidden_states(image_path):
    """Return every vision hidden state on CPU, retaining all tokens."""
    image_tensor = to_tensor(Image.open(image_path)).to(DEVICE, DTYPE)
    pixel_values = preprocess(image_tensor).unsqueeze(0)
    with torch.inference_mode():
        outputs = vision_tower(pixel_values, output_hidden_states=True)
    return tuple(state.detach().float().cpu() for state in outputs.hidden_states)

# Process one image at a time and immediately move its states to CPU.
hidden_states = {
    name: extract_hidden_states(path) for name, path in IMAGE_PATHS.items()
}

counts = {name: len(states) for name, states in hidden_states.items()}
assert len(set(counts.values())) == 1, counts
shapes = {
    name: [tuple(state.shape) for state in states]
    for name, states in hidden_states.items()
}
assert shapes["source"] == shapes["target"] == shapes["adversarial"]

vision_config = vision_tower.config
architecture = {
    "model_type": vision_config.model_type,
    "encoder_layers": vision_config.num_hidden_layers,
    "returned_hidden_states": counts["source"],
    "embedding_state_included": counts["source"] == vision_config.num_hidden_layers + 1,
    "hidden_state_shape": shapes["source"][0],
    "amp_hidden_states[-2]_index": counts["source"] - 2,
    "amp_hidden_states[-2]_shape": shapes["source"][-2],
}
architecture

In [ ]:
def flattened_cosine(left, right):
    return F.cosine_similarity(left.reshape(1, -1), right.reshape(1, -1)).item()

rows = []
for layer, (source, target, adversarial) in enumerate(
    zip(
        hidden_states["source"],
        hidden_states["target"],
        hidden_states["adversarial"],
    )
):
    R = flattened_cosine(adversarial, source)
    T = flattened_cosine(adversarial, target)
    B = flattened_cosine(source, target)
    rows.append(
        {
            "layer": layer,
            "R": R,
            "T": T,
            "B": B,
            "G": T - B,
            "tensor_shape": str(tuple(source.shape)),
        }
    )

results = pd.DataFrame(rows, columns=["layer", "R", "T", "B", "G", "tensor_shape"])
results.to_csv(OUTPUT_CSV, index=False)
results